## Exercise 1: Set Up the AutoSphere Data Platform

You are a data engineer at **AutoSphere AG**, a global automotive manufacturer. Your connected vehicle platform ingests telemetry from millions of vehicles, manages customer registration data, and tracks service history.

Before you can apply governance controls, you need to create the catalog, schema, and tables that form the foundation of the platform. You'll also add **comments** to document the purpose of each table and column — making your data assets discoverable by other teams.

### Setup — Create the catalog, schema, and tables

To keep the focus on governance controls, the AutoSphere data platform is set up for you in the cell below. It creates:

- **`automotive_catalog_sp`** — the Unity Catalog catalog for the AutoSphere platform, with a descriptive comment
- **`automotive_catalog_sp.governance_lab`** — the schema used throughout this lab
- **`customer_registrations`** — one row per registered customer and their associated vehicle; includes a table-level comment
- **`vehicle_telemetry`** — continuous telemetry events from connected vehicles; includes column-level comments on `speed_kmh` and `battery_level_pct`

Run the cell to prepare your environment, then continue to Exercise 2.

In [0]:
# Since no default storage is enabled, we are inheriting the storage path from the default catalog's root.
# Use the current catalog to reliably find the workspace default catalog,
# regardless of its naming convention.
default_catalog = spark.catalog.currentCatalog()

storage_root = (
    spark.sql(f"DESCRIBE CATALOG EXTENDED {default_catalog}")
    .filter("info_name = 'Storage Root'")
    .select("info_value")
    .first()[0]
)
print (f"Storage root: {storage_root}")

spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS automotive_catalog_sp
    MANAGED LOCATION '{storage_root}'
    COMMENT 'Connected vehicle platform catalog for AutoSphere AG — stores registration, telemetry, and service data.'
""")

Storage root: abfss://unity-catalog-storage@dbstoragejnsxrbyseoy6a.dfs.core.windows.net/7405607263146590


DataFrame[]

In [0]:
%sql
use catalog automotive_catalog_sp;
select current_catalog()

current_catalog()
automotive_catalog_sp


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS automotive_catalog_sp.governance_lab
  COMMENT 'Governance lab schema for hands-on Unity Catalog exercises.';

-- Create the customer_registrations table
CREATE TABLE IF NOT EXISTS automotive_catalog_sp.governance_lab.customer_registrations
  COMMENT 'One row per registered AutoSphere customer and their associated vehicle.'
AS SELECT * FROM VALUES
  (1, 'Lukas Bauer',    'lukas.bauer@autosphere.de',   'DE-LB-4821', 'DE', 'VH-001'),
  (2, 'Sophie Martin',  'sophie.martin@autosphere.fr',  'FR-SM-9034', 'FR', 'VH-002'),
  (3, 'James Clarke',   'james.clarke@autosphere.uk',   'UK-JC-1172', 'UK', 'VH-003'),
  (4, 'Yuki Tanaka',    'yuki.tanaka@autosphere.jp',    'JP-YT-5561', 'JP', 'VH-004'),
  (5, 'Maria Santos',   'maria.santos@autosphere.pt',   'PT-MS-8843', 'PT', 'VH-005'),
  (6, 'Carlos Rivera',  'carlos.rivera@autosphere.es',  'ES-CR-3309', 'ES', 'VH-006')
AS t(customer_id, full_name, email, driver_license_no, country, vehicle_id);

-- Create the vehicle_telemetry table
CREATE TABLE IF NOT EXISTS automotive_catalog_sp.governance_lab.vehicle_telemetry (
  event_id          BIGINT    COMMENT 'Unique telemetry event identifier',
  vehicle_id        STRING,
  event_time        TIMESTAMP,
  speed_kmh         INT       COMMENT 'Instantaneous vehicle speed in kilometres per hour',
  battery_level_pct INT       COMMENT 'State of charge as a percentage (0–100)',
  latitude          DOUBLE,
  longitude         DOUBLE,
  country           STRING
);

INSERT INTO automotive_catalog_sp.governance_lab.vehicle_telemetry VALUES
  (1, 'VH-001', TIMESTAMP '2026-03-01 08:15:00', 112, 78,  48.8566,   2.3522, 'DE'),
  (2, 'VH-002', TIMESTAMP '2026-03-01 08:16:00', 95,  55,  51.5074,  -0.1278, 'FR'),
  (3, 'VH-003', TIMESTAMP '2026-03-01 08:17:00', 130, 91,  40.4168,  -3.7038, 'UK'),
  (4, 'VH-004', TIMESTAMP '2026-03-01 08:18:00', 88,  34,  35.6762, 139.6503, 'JP'),
  (5, 'VH-005', TIMESTAMP '2026-03-01 08:19:00', 105, 62,  38.7223,  -9.1393, 'PT'),
  (6, 'VH-006', TIMESTAMP '2026-03-01 08:20:00', 77,  47,  41.3851,   2.1734, 'ES');

num_affected_rows,num_inserted_rows
6,6


## Exercise 2: Tag Data Assets for Governance

AutoSphere's compliance team requires that all tables and columns containing **personally identifiable information (PII)** are consistently labelled. Tags enable automated discovery and will later drive access policies.

In this exercise, you apply table-level and column-level tags to classify the `customer_registrations` table.

### Task 2.1 — Add table-level tags

Tag the `customer_registrations` table to classify it by domain and sensitivity:

- `domain` = `customer`
- `data_classification` = `confidential`

> 🤖 **Genie Code tip:** Ask:
> *"How do I add key-value tags to a Unity Catalog table in Databricks SQL?"*

**Hint:** Use `ALTER TABLE ... SET TAGS ('key' = 'value', 'key2' = 'value2')`.

In [0]:
%sql
ALTER TABLE automotive_catalog_sp.governance_lab.customer_registrations
SET TAGS ('domain' = 'customer', 'data_classification' = 'confidential');

### Task 2.2 — Add column-level PII tags

Tag the PII columns in `customer_registrations` to identify them as sensitive:

- Column `email` → `pii` = `email`
- Column `driver_license_no` → `pii` = `driver_license`

> 🤖 **Genie Code tip:** Ask:
> *"How do I add a tag to a specific column in a Unity Catalog table using SQL in Databricks?"*

**Hint:** Use `ALTER TABLE ... ALTER COLUMN <col> SET TAGS (...)`.

In [0]:
%sql
-- Tag the email column with pii = email
ALTER TABLE automotive_catalog_sp.governance_lab.customer_registrations
ALTER COLUMN email SET TAGS ('pii' = 'email');

-- Tag the driver_license_no column with pii = driver_license
ALTER TABLE automotive_catalog_sp.governance_lab.customer_registrations
ALTER COLUMN driver_license_no SET TAGS ('pii' = 'driver_license');

### Task 2.3 — Verify the tags

Use `system.information_schema.table_tags` to confirm the table-level tags have been applied. Then describe the columns to confirm the column tags are present. Use `system.information_schema.column_tags` for this.

> 🤖 **Genie Code tip:** Ask:
> *"How can I verify tags on a Unity Catalog table and its columns in Databricks SQL?"*

In [0]:
%sql
-- Show table-level tags for customer_registrations
SELECT
  catalog_name,
  schema_name,
  table_name,
  tag_name,
  tag_value
FROM system.information_schema.table_tags
WHERE catalog_name = 'automotive_catalog_sp'
  AND schema_name = 'governance_lab'
  AND table_name = 'customer_registrations'
ORDER BY tag_name;

catalog_name,schema_name,table_name,tag_name,tag_value
automotive_catalog_sp,governance_lab,customer_registrations,data_classification,confidential
automotive_catalog_sp,governance_lab,customer_registrations,domain,customer


In [0]:
%sql
-- Show column-level tags for customer_registrations
SELECT
  catalog_name,
  schema_name,
  table_name,
  column_name,
  tag_name,
  tag_value
FROM system.information_schema.column_tags
WHERE catalog_name = 'automotive_catalog_sp'
  AND schema_name = 'governance_lab'
  AND table_name = 'customer_registrations'
ORDER BY column_name, tag_name;

catalog_name,schema_name,table_name,column_name,tag_name,tag_value
automotive_catalog_sp,governance_lab,customer_registrations,driver_license_no,pii,driver_license
automotive_catalog_sp,governance_lab,customer_registrations,email,pii,email


## Exercise 3: Configure Data Retention

AutoSphere's vehicle telemetry table is written to continuously. To comply with GDPR data minimisation principles and control storage costs, the platform must:

- Retain only **14 days** of table history.
- Permanently remove old data files using `VACUUM`.
- Enable **predictive optimization** so that future maintenance runs automatically.

In this exercise, you configure retention settings, simulate record deletion, and run a `VACUUM` operation.

### Task 3.1 — Configure retention properties

Set the following Delta Lake table properties on `vehicle_telemetry`:

- `delta.logRetentionDuration` = `interval 14 days`
- `delta.deletedFileRetentionDuration` = `interval 14 days`

> 🤖 **Genie Code tip:** Ask:
> *"How do I set Delta Lake retention duration properties on a Unity Catalog table in Databricks?"*

**Hint:** Use `ALTER TABLE ... SET TBLPROPERTIES (...)`.

In [0]:
%sql
-- Set logRetentionDuration and deletedFileRetentionDuration to 14 days
-- on automotive_catalog_sp.governance_lab.vehicle_telemetry
ALTER TABLE automotive_catalog_sp.governance_lab.vehicle_telemetry
SET TBLPROPERTIES (
  'delta.logRetentionDuration' = 'interval 14 days',
  'delta.deletedFileRetentionDuration' = 'interval 14 days'
);

### Task 3.2 — Simulate a GDPR deletion request

A customer has submitted a right-to-erasure request. Delete all telemetry records for vehicle `VH-003` from the `vehicle_telemetry` table.

After deletion, verify the records are gone by querying the table.

> 🤖 **Genie Code tip:** Ask:
> *"How do I delete specific rows from a Delta Lake table in Databricks SQL?"*

In [0]:
%sql
-- Delete all rows for vehicle_id = 'VH-003' from vehicle_telemetry
DELETE FROM automotive_catalog_sp.governance_lab.vehicle_telemetry
WHERE vehicle_id = 'VH-003';

-- Verify the deletion by querying the table
SELECT * FROM automotive_catalog_sp.governance_lab.vehicle_telemetry
ORDER BY event_id;

event_id,vehicle_id,event_time,speed_kmh,battery_level_pct,latitude,longitude,country
1,VH-001,2026-03-01T08:15:00.000Z,112,78,48.8566,2.3522,DE
2,VH-002,2026-03-01T08:16:00.000Z,95,55,51.5074,-0.1278,FR
4,VH-004,2026-03-01T08:18:00.000Z,88,34,35.6762,139.6503,JP
5,VH-005,2026-03-01T08:19:00.000Z,105,62,38.7223,-9.1393,PT
6,VH-006,2026-03-01T08:20:00.000Z,77,47,41.3851,2.1734,ES


### Task 3.3 — Run VACUUM to purge deleted data files

The deleted rows still exist in underlying Parquet files until `VACUUM` is run. Execute `VACUUM` on `vehicle_telemetry` to permanently remove files that are no longer referenced by any current or recent table version.

Use the default retention of **168 hours (7 days)**. This is safe, works on Serverless compute, and is appropriate for a lab environment.

> 🤖 **Genie Code tip:** Ask:
> *"How do I run VACUUM on a Delta Lake table in Databricks, and what does the RETAIN option do?"*

**Note:** On Serverless compute, overriding the minimum retention duration is not supported. Always specify a `RETAIN` value of at least 168 hours, or omit `RETAIN` entirely to use the default.

In [0]:
%sql
-- VACUUM removes data files that are no longer referenced by the current table version.
-- The RETAIN option specifies the minimum age (in hours) of files to delete.
-- Files newer than the retention threshold are kept, even if unreferenced.
-- On Serverless compute, RETAIN must be at least 168 hours (7 days).

VACUUM automotive_catalog_sp.governance_lab.vehicle_telemetry RETAIN 168 HOURS

path
abfss://unity-catalog-storage@dbstoragejnsxrbyseoy6a.dfs.core.windows.net/7405607263146590/__unitystorage/catalogs/a81c617c-1fde-4e1a-bcb3-a3d57bd22d20/tables/9394a5e1-5879-4ee3-85bb-c232808b5d47


### Task 3.4 — Enable predictive optimization

Enable **predictive optimization** on the `governance_lab` schema so that future `VACUUM` and `OPTIMIZE` maintenance runs automatically without manual scheduling.

> 🤖 **Genie Code tip:** Ask:
> *"How do I enable predictive optimization on a Unity Catalog schema in Databricks?"*

**Hint:** Use `ALTER SCHEMA ... ENABLE PREDICTIVE OPTIMIZATION`.

In [0]:
%sql
ALTER SCHEMA automotive_catalog_sp.governance_lab ENABLE PREDICTIVE OPTIMIZATION

## Exercise 4: Query Data Lineage Programmatically

> 📋 **Before running this exercise**: If you haven't already done so, pause here and follow the **Catalog Explorer lineage steps** in the lab setup instructions to explore lineage visually in the UI. Then return here to query lineage data using SQL.

Unity Catalog captures all read and write events and stores them in the `system.access.table_lineage` system table. This lets you programmatically answer questions like: *"Which tables in our automotive catalog were most queried this week?"*

### Task 4.1 — Find recently accessed tables in your catalog

Query `system.access.table_lineage` to find all **write events** (source tables) within the past 7 days that relate to `automotive_catalog_sp`. Return the table name and the count of distinct events, ordered by the most active tables first.

> 🤖 **Genie Code tip:** Ask:
> *"How do I query system.access.table_lineage to find recently accessed Unity Catalog tables in Databricks?"*

**Hint:** Filter on `source_table_full_name LIKE 'automotive_catalog_sp%'` and group by table name.

In [0]:
%sql
SELECT
  source_table_full_name,
  COUNT(*) AS event_count
FROM system.access.table_lineage
WHERE source_table_full_name LIKE 'automotive_catalog_sp%'
  AND event_time > current_timestamp() - INTERVAL 7 DAYS
GROUP BY source_table_full_name
ORDER BY event_count DESC

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8758407138062768>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "SELECT\n  source_table_full_name,\n  COUNT(*) AS event_count\nFROM system.access.table_lineage\nWHERE source_table_full_name LIKE 'automotive_catalog_sp%'\n  AND event_time > current_timestamp() - INTERVAL 7 DAYS\nGROUP BY source_table_full_name\nORDER BY event_count DESC\n")

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if get

### Task 4.2 — View the history of vehicle_telemetry

Use `DESCRIBE HISTORY` to see the full Delta Lake operation history for `vehicle_telemetry`. Identify the version where the deletion occurred (`DELETE` operation).

> 🤖 **Genie Code tip:** Ask:
> *"How do I view the full version history of a Delta Lake table in Databricks SQL?"*

In [0]:
%sql
DESCRIBE HISTORY automotive_catalog_sp.governance_lab.vehicle_telemetry

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
6,2026-09-01T10:54:54.001Z,142797373781581,user1-64638608@lodsprodmca.onmicrosoft.com,VACUUM END,Map(status -> COMPLETED),null,List(1695782247662219),419de90d-3ce3-4c70-91d3-22dbeb018dc1,0901-101849-m1ozz0do-v2n,5,SnapshotIsolation,true,"Map(numDeletedFiles -> 0, numVacuumedDirectories -> 1)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
5,2026-09-01T10:54:54.000Z,142797373781581,user1-64638608@lodsprodmca.onmicrosoft.com,VACUUM START,"Map(retentionCheckEnabled -> true, defaultRetentionMillis -> 1209600000)",null,List(1695782247662219),419de90d-3ce3-4c70-91d3-22dbeb018dc1,0901-101849-m1ozz0do-v2n,4,SnapshotIsolation,true,"Map(numFilesToDelete -> 0, sizeOfDataToDelete -> 0)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
4,2026-09-01T10:53:24.000Z,142797373781581,user1-64638608@lodsprodmca.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1695782247662219),2888aa52-1ae9-411f-b2e3-aa2d51112755,0901-101849-m1ozz0do-v2n,3,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 2680, p25FileSize -> 3542, numDeletionVectorsRemoved -> 1, minFileSize -> 3542, numAddedFiles -> 1, maxFileSize -> 3542, p75FileSize -> 3542, p50FileSize -> 3542, numAddedBytes -> 3542)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
3,2026-09-01T10:53:22.000Z,142797373781581,user1-64638608@lodsprodmca.onmicrosoft.com,DELETE,"Map(predicate -> [""(vehicle_id#12147 = VH-003)""])",null,List(1695782247662219),2888aa52-1ae9-411f-b2e3-aa2d51112755,0901-101849-m1ozz0do-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2839, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 2173, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 652)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
2,2026-09-01T10:52:34.000Z,142797373781581,user1-64638608@lodsprodmca.onmicrosoft.com,SET TBLPROPERTIES,"Map(properties -> {""delta.logRetentionDuration"":""interval 14 days"",""delta.deletedFileRetentionDuration"":""interval 14 days""})",null,List(1695782247662219),4fe0aa28-3de9-4458-994f-8cae56d28a1e,0901-101849-m1ozz0do-v2n,1,WriteSerializable,true,Map(),null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
1,2026-09-01T10:42:20.000Z,142797373781581,user1-64638608@lodsprodmca.onmicrosoft.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(1695782247662219),95eebd76-7a1c-4bda-b20a-1e5c3f03451d,0901-101849-m1ozz0do-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 6, numOutputBytes -> 2680)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
0,2026-09-01T10:42:18.000Z,142797373781581,user1-64638608@lodsprodmca.onmicrosoft.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""databricks.internal.autoUpgrades.delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""io.unitycatalog.tableId"":""9394a5e1-5879-4ee3-85bb-c232808b5d47"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-7ec2028f-4047-4004-a090-e546605724de"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-a9bba999-8ba2-4dc6-bb65-3078207d173a""}, statsOnLoad -> false)",null,List(1695782247662219),2dbe6156-cffa-48b3-b0ff-b61fa058bd07,0901-101849-m1ozz0do-v2n,null,WriteSerializable,true,Map(

## Exercise 5: Audit Logging

AutoSphere's compliance team needs to prove to regulators that only authorised users accessed customer data during the past 30 days. The team also wants to be alerted whenever permissions are modified on the Unity Catalog metastore.

You will use the `system.access.audit` system table to answer both of these requirements.

### Task 5.1 — Find who accessed the customer_registrations table

Query `system.access.audit` to list all users who accessed `automotive_catalog_sp.governance_lab.customer_registrations` in the past 30 days. Include the action name and the time of each access, ordered by most recent first.

> 🤖 **Genie Code tip:** Ask:
> *"How do I query the Databricks audit log system table to find who accessed a specific Unity Catalog table?"*

**Hint:** Filter on `request_params.full_name_arg` and `action_name IN ('getTable', 'createTable')`. Use `user_identity.email` for the user column.

In [0]:
%sql
-- Find who accessed the customer_registrations table in the past 30 days
SELECT
  user_identity.email AS user_email,
  action_name,
  event_time
FROM system.access.audit
WHERE action_name IN ('getTable', 'createTable')
  AND request_params.full_name_arg = 'automotive_catalog_sp.governance_lab.customer_registrations'
  AND event_time > current_timestamp() - INTERVAL 30 DAYS
ORDER BY event_time DESC

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8758407138062773>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "-- Find who accessed the customer_registrations table in the past 30 days\nSELECT\n  user_identity.email AS user_email,\n  action_name,\n  event_time\nFROM system.access.audit\nWHERE action_name IN ('getTable', 'createTable')\n  AND request_params.full_name_arg = 'automotive_catalog_sp.governance_lab.customer_registrations'\n  AND event_time > current_timestamp() - INTERVAL 30 DAYS\nORDER BY event_time DESC\n")

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 

### Task 5.2 — Detect permission changes across your catalog

Query `system.access.audit` to list all **permission change events** (`updatePermissions` action) on securable objects in `automotive_catalog_sp` during the past 30 days. Include who made the change and what object was affected.

> 🤖 **Genie Code tip:** Ask:
> *"How do I find all permission changes in the Databricks audit log for a specific Unity Catalog catalog?"*

**Hint:** Filter on `service_name = 'unityCatalog'`, `action_name = 'updatePermissions'`, and `request_params.securable_full_name LIKE 'automotive_catalog_sp%'`.

In [0]:
%sql
-- Detect permission changes on automotive_catalog_sp objects in the past 30 days
SELECT
  event_time,
  user_identity.email AS user_email,
  request_params.securable_type,
  request_params.securable_full_name,
  request_params.changes
FROM system.access.audit
WHERE service_name = 'unityCatalog'
  AND action_name = 'updatePermissions'
  AND request_params.securable_full_name LIKE 'automotive_catalog_sp%'
  AND event_time > current_timestamp() - INTERVAL 30 DAYS
ORDER BY event_time DESC

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8758407138062775>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "-- Detect permission changes on automotive_catalog_sp objects in the past 30 days\nSELECT\n  event_time,\n  user_identity.email AS user_email,\n  request_params.securable_type,\n  request_params.securable_full_name,\n  request_params.changes\nFROM system.access.audit\nWHERE service_name = 'unityCatalog'\n  AND action_name = 'updatePermissions'\n  AND request_params.securable_full_name LIKE 'automotive_catalog_sp%'\n  AND event_time > current_timestamp() - INTERVAL 30 DAYS\nORDER BY event_time DESC\n")

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn

### Task 5.3 — Summarise audit activity by user

Create a summary query that counts the total number of audit events per user for Unity Catalog actions in the past 7 days. This gives the compliance team a quick overview of who was most active on the platform.

> 🤖 **Genie Code tip:** Ask:
> *"How do I aggregate the Databricks audit log to count events per user for Unity Catalog activity?"*

In [0]:
%sql
-- Count total audit events per user for Unity Catalog activity in the last 7 days
SELECT
  user_identity.email AS user_email,
  COUNT(*) AS event_count
FROM system.access.audit
WHERE service_name = 'unityCatalog'
  AND event_time > current_timestamp() - INTERVAL 7 DAYS
GROUP BY user_identity.email
ORDER BY event_count DESC

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8758407138062777>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "-- Count total audit events per user for Unity Catalog activity in the last 7 days\nSELECT\n  user_identity.email AS user_email,\n  COUNT(*) AS event_count\nFROM system.access.audit\nWHERE service_name = 'unityCatalog'\n  AND event_time > current_timestamp() - INTERVAL 7 DAYS\nGROUP BY user_identity.email\nORDER BY event_count DESC\n")

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the